# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities using their `@id` fields as required.

### Dataset Source
The dataset source is provided via the Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

We will use this schema to load metadata and records, inspect available fields and record sets, and apply basic exploratory data analysis using proper references to entity `@id` values.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
We load the dataset metadata and records with `mlcroissant`, referencing the provided Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and print overview (access as object, not dict)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata # treat as object, not dict
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review the available record sets and their fields, referencing all by their `@id`.

We will list the record sets, fields, and columns using their `@id` values, and show sample records from the primary record set.

In [ ]:
# Get all record sets defined in the dataset, referenced by @id
record_sets = list(dataset.record_sets.keys())
print("Available record sets (by @id):")
for rs_id in record_sets:
    print(f"  - {rs_id}")

# Display fields and columns for each record set by @id
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"\nRecord Set @id: {rs_id}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for f in rs.fields:
            print(f"    - @id: {getattr(f, '@id', '?')}, name: {getattr(f, 'name', '?')}, dataType: {getattr(f, 'dataType', '?')}")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for col in rs.columns:
            print(f"    - @id: {getattr(col, '@id', '?')}, name: {getattr(col, 'name', '?')}, dataType: {getattr(col, 'dataType', '?')}")

# Show several sample records using the main record set
main_record_set_id = record_sets[0]  # Assume the primary record set is first
print(f"\nSample records from record set @id {main_record_set_id}:")
for record in dataset.records(record_set=main_record_set_id):
    pprint(record)
    break  # Show just one for overview (remove break for more records)

## 3. Data Extraction
We load the tabular data for analysis from record sets using their `@id`. This section extracts all records for each available record set and for key fields referenced by their `@id`. The data will be loaded into pandas DataFrames for further exploration.

In [ ]:
# Get all records for each record set by @id
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded DataFrame for record set {rs_id}: columns are\n{df.columns.tolist()}")
    print(f"Sample rows from {rs_id}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
We now process the tabular data for basic analysis. This includes filtering numeric fields, normalization, and grouping. All fields are referenced by their `@id`, as required.

For illustration:
- We select a numeric field from the dataset referenced by its `@id` (for example, 'age').
- We filter rows with values above a chosen threshold.
- We normalize this numeric field.
- Optionally, we group by a categorical field referenced by `@id` (e.g., 'sex').

In [ ]:
# Choose the main record set and inspect available numeric fields
df_main = dataframes[main_record_set_id]

# Let's try to identify numeric columns. We'll select one (e.g., '@id: age')
numeric_field_ids = [col for col in df_main.columns if 'age' in col.lower() or 'interval' in col.lower()]
print(f"Numeric fields candidate IDs: {numeric_field_ids}")
numeric_field_id = numeric_field_ids[0] if numeric_field_ids else df_main.select_dtypes(include=['int64', 'float64']).columns[0]
print(f"Selected numeric field for analysis: {numeric_field_id}")

# Filtering
threshold = 10
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: check for groupable fields (e.g., sex, anatomical location) by their @id
group_field_candidates = [col for col in df_main.columns if 'sex' in col.lower() or 'anatomical' in col.lower()]
group_field_id = group_field_candidates[0] if group_field_candidates else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}: (mean of {numeric_field_id})")
    display(grouped_df.head())

## 5. Visualization
Visualize numeric field distribution and relationship with a key categorical variable, referencing all fields by their `@id`.

In [ ]:
# Plot histogram of the numeric field for the main record set
plt.figure(figsize=(8, 5))
df_main[numeric_field_id].hist(bins=15, edgecolor='black')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

# If grouping field is available, plot mean numeric value by group
if group_field_id:
    grouped_df = df_main.groupby(group_field_id)[numeric_field_id].mean()
    grouped_df.plot(kind='bar', figsize=(8, 5))
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(group_field_id)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
We explored the FAIR² dataset, referencing all entities by their `@id` using the `mlcroissant` library.

Key findings:
- The dataset provides detailed clinicopathological and molecular characteristics of second primary colorectal cancer in survivors.
- Record sets, fields, and columns were successfully listed and extracted by `@id`.
- Basic filtering, normalization, grouping, and visualization operations were demonstrated.
- This notebook provides a reproducible workflow for FAIR data exploration in oncology with Croissant-compliant schemas.

Further analysis can continue with advanced statistical or machine learning workflows, always referencing entities by their unique `@id`.